In [1]:
from pathlib import Path

import pandas as pd


# Si el notebook está dentro de /notebooks, subimos un nivel.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "instacart"

ORDERS_PATH = RAW_DATA_DIR / "orders.csv"
ORDER_PRODUCTS_PRIOR_PATH = RAW_DATA_DIR / "order_products__prior.csv"
ORDER_PRODUCTS_TRAIN_PATH = RAW_DATA_DIR / "order_products__train.csv"
PRODUCTS_PATH = RAW_DATA_DIR / "products.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data dir:", RAW_DATA_DIR)

Project root: c:\Users\Lenovo\Desktop\retail-recommender-mlops
Raw data dir: c:\Users\Lenovo\Desktop\retail-recommender-mlops\data\raw\instacart


In [2]:
orders = pd.read_csv(ORDERS_PATH)
products = pd.read_csv(PRODUCTS_PATH)

print("orders:", orders.shape)
print("products:", products.shape)

orders.head()

orders: (3421083, 7)
products: (49688, 4)


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [3]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                str    
 3   order_number            int64  
 4   order_dow               int64  
 5   order_hour_of_day       int64  
 6   days_since_prior_order  float64
dtypes: float64(1), int64(5), str(1)
memory usage: 198.9 MB


In [4]:
orders["eval_set"].value_counts()

eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64

In [5]:
#Seleccionar usuarios con ordenes en el conjunto de entrenamiento
train_orders = orders[orders["eval_set"] == "train"].copy()

print(train_orders.shape)
train_orders.head()

(131209, 7)


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
10,1187899,1,train,11,4,8,14.0
25,1492625,2,train,15,1,11,30.0
49,2196797,5,train,5,0,11,6.0
74,525192,7,train,21,2,11,6.0
78,880375,8,train,4,1,14,10.0


In [6]:
train_user_ids = train_orders["user_id"].unique()

print("Usuarios con orden train:", len(train_user_ids))
print(train_user_ids[:10])

Usuarios con orden train: 131209
[ 1  2  5  7  8  9 10 13 14 17]


In [7]:
# Tomar una muestra pequeña de usuarios para acelerar el desarrollo

SAMPLE_USERS = 500

sample_user_ids = pd.Series(train_user_ids).sample(
    n=SAMPLE_USERS,
    random_state=42
).tolist()

len(sample_user_ids)

500

In [8]:
# Filtrar las ordenes para quedarnos solo con las de los usuarios seleccionados

sample_orders = orders[orders["user_id"].isin(sample_user_ids)].copy()

print(sample_orders.shape)
sample_orders.head()

(9357, 7)


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
8149,157924,502,prior,1,2,14,NaN
8150,1347839,502,prior,2,6,11,11.0
8151,149026,502,prior,3,5,11,13.0
8152,3192899,502,prior,4,4,11,6.0
8153,2770330,502,prior,5,0,14,10.0


In [9]:
sample_orders["eval_set"].value_counts()

eval_set
prior    8857
train     500
Name: count, dtype: int64

In [10]:
## Separar las ordenes de train y prior

sample_prior_orders = sample_orders[sample_orders["eval_set"] == "prior"].copy()
sample_train_orders = sample_orders[sample_orders["eval_set"] == "train"].copy()

print("prior orders:", sample_prior_orders.shape)
print("train orders:", sample_train_orders.shape)

prior orders: (8857, 7)
train orders: (500, 7)


In [11]:
# Cargar los productos comprados por estos usuarios en el conjunto de entrenamiento

prior_order_ids = set(sample_prior_orders["order_id"])
train_order_ids = set(sample_train_orders["order_id"])

order_products_prior = pd.read_csv(ORDER_PRODUCTS_PRIOR_PATH)
order_products_train = pd.read_csv(ORDER_PRODUCTS_TRAIN_PATH)

sample_order_products_prior = order_products_prior[
    order_products_prior["order_id"].isin(prior_order_ids)
].copy()

sample_order_products_train = order_products_train[
    order_products_train["order_id"].isin(train_order_ids)
].copy()

print("sample_order_products_prior:", sample_order_products_prior.shape)
print("sample_order_products_train:", sample_order_products_train.shape)

sample_order_products_prior: (89280, 4)
sample_order_products_train: (5341, 4)


### Unir productos con usuarios

order_products_prior solo tiene order_id, no user_id. Por eso lo cruzamos con sample_prior_orders.

In [12]:
prior_df = (
    sample_order_products_prior
    .merge(
        sample_prior_orders[
            [
                "order_id",
                "user_id",
                "order_number",
                "order_dow",
                "order_hour_of_day",
                "days_since_prior_order",
            ]
        ],
        on="order_id",
        how="left"
    )
)

train_df = (
    sample_order_products_train
    .merge(
        sample_train_orders[
            [
                "order_id",
                "user_id",
                "order_number",
                "order_dow",
                "order_hour_of_day",
                "days_since_prior_order",
            ]
        ],
        on="order_id",
        how="left"
    )
)

print("prior_df:", prior_df.shape)
print("train_df:", train_df.shape)

prior_df.head()

prior_df: (89280, 9)
train_df: (5341, 9)


,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,595,30027,1,1,32240,68,4,6,3.0
1,595,44144,2,1,32240,68,4,6,3.0
2,595,22713,3,1,32240,68,4,6,3.0
3,595,40852,4,1,32240,68,4,6,3.0
4,595,19508,5,1,32240,68,4,6,3.0


In [13]:
# ejemplo de un usuario con varias ordenes
sample_user = sample_user_ids[0]

print("User:", sample_user)

prior_df[prior_df["user_id"] == sample_user].head(20)

User: 181364


,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
32331,1250243,16398,1,0,181364,4,3,8,30.0
32332,1250243,17461,2,0,181364,4,3,8,30.0
32333,1250243,29592,3,0,181364,4,3,8,30.0
32334,1250243,47766,4,0,181364,4,3,8,30.0
32335,1250243,30363,5,0,181364,4,3,8,30.0
32336,1250243,21791,6,0,181364,4,3,8,30.0
32337,1250243,37029,7,0,181364,4,3,8,30.0
40934,1555935,44702,1,0,181364,3,2,9,30.0
40935,1555935,18465,2,1,181364,3,2,9,30.0
40936,1555935,49683,3,0,181364,3,2,9,30.0


In [16]:
train_df[train_df["user_id"] == sample_user]

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
4282,2670051,39928,1,1,181364,7,4,13,30.0


Ahora construimos el dataset supervisado. La unidad de análisis será:

usuario + producto candidato

Es decir: para cada usuario, tomamos los productos que compró en su historial prior y preguntamos si los volvió a comprar en su orden train.

In [21]:
# Cada candidato es una combinación user_id + product_id.
# Por ahora solo usamos productos que el usuario ya compró antes.
# Esto convierte el problema en:
# "De los productos que el usuario ya conoce, ¿cuáles volverá a comprar?"

candidates = (
    prior_df[["user_id", "product_id"]]
    .drop_duplicates()
    .copy()
)

print(candidates.shape)
candidates.head()

(34508, 2)


,user_id,product_id
0,32240,30027
1,32240,44144
2,32240,22713
3,32240,40852
4,32240,19508


In [22]:
## Crear el target

# En train_df están los productos comprados en la siguiente orden conocida.
# Si un user_id + product_id aparece en train_df, entonces target = 1.
# Si no aparece, target = 0.

target_products = (
    train_df[["user_id", "product_id"]]
    .drop_duplicates()
    .copy()
)

target_products["target"] = 1

target_products.head()

,user_id,product_id,target
0,42756,11913,1
1,42756,18159,1
2,42756,4461,1
3,42756,21616,1
4,42756,23622,1


In [23]:
training_data = (
    candidates
    .merge(
        target_products,
        on=["user_id", "product_id"],
        how="left"
    )
)

training_data["target"] = training_data["target"].fillna(0).astype(int)

print(training_data.shape)
training_data.head()

(34508, 3)


,user_id,product_id,target
0,32240,30027,1
1,32240,44144,1
2,32240,22713,1
3,32240,40852,1
4,32240,19508,0


In [24]:
# Revisar el balance de clases

training_data["target"].value_counts(normalize=True)

target
0    0.907761
1    0.092239
Name: proportion, dtype: float64

In [25]:
# Cuántas veces compró cada usuario cada producto en su historial prior.

user_product_features = (
    prior_df
    .groupby(["user_id", "product_id"])
    .agg(
        user_product_purchase_count=("order_id", "nunique"),
        user_product_reorder_count=("reordered", "sum"),
        user_product_avg_add_to_cart_order=("add_to_cart_order", "mean"),
        user_product_last_order_number=("order_number", "max"),
    )
    .reset_index()
)

user_product_features.head()

,user_id,product_id,user_product_purchase_count,user_product_reorder_count,user_product_avg_add_to_cart_order,user_product_last_order_number
0,502,283,1,0,18.0,4
1,502,1463,1,0,5.0,4
2,502,2747,1,0,20.0,4
3,502,3712,1,0,15.0,2
4,502,4853,2,1,16.5,3


In [26]:
# Cuántas órdenes prior tiene cada usuario.
# Sirve para contextualizar la frecuencia:
# comprar un producto 3 veces no significa lo mismo en 4 órdenes que en 40 órdenes.

user_features = (
    sample_prior_orders
    .groupby("user_id")
    .agg(
        user_total_prior_orders=("order_id", "nunique"),
        user_avg_days_since_prior_order=("days_since_prior_order", "mean"),
    )
    .reset_index()
)

user_features.head()

,user_id,user_total_prior_orders,user_avg_days_since_prior_order
0,502,7,15.500000
1,519,10,16.000000
2,866,22,16.095238
3,899,10,2.555556
4,956,11,20.600000


### Popularidad global del producto

Popularidad del producto en la muestra.
Mide qué tan común es el producto entre todos los usuarios.

In [27]:
product_features = (
    prior_df
    .groupby("product_id")
    .agg(
        product_purchase_count=("order_id", "nunique"),
        product_reorder_count=("reordered", "sum"),
        product_avg_add_to_cart_order=("add_to_cart_order", "mean"),
    )
    .reset_index()
)

product_features["product_reorder_rate"] = (
    product_features["product_reorder_count"] /
    product_features["product_purchase_count"]
)

product_features.head()

,product_id,product_purchase_count,product_reorder_count,product_avg_add_to_cart_order,product_reorder_rate
0,4,1,0,5.000000,0.000000
1,8,5,4,24.600000,0.800000
2,10,19,14,9.157895,0.736842
3,18,15,14,2.133333,0.933333
4,25,2,0,13.500000,0.000000


In [28]:
### Combinar features con el dataset supervisado

model_df = (
    training_data
    .merge(
        user_product_features,
        on=["user_id", "product_id"],
        how="left"
    )
    .merge(
        user_features,
        on="user_id",
        how="left"
    )
    .merge(
        product_features,
        on="product_id",
        how="left"
    )
)

print(model_df.shape)
model_df.head()

(34508, 13)


,user_id,product_id,target,user_product_purchase_count,user_product_reorder_count,user_product_avg_add_to_cart_order,user_product_last_order_number,user_total_prior_orders,user_avg_days_since_prior_order,product_purchase_count,product_reorder_count,product_avg_add_to_cart_order,product_reorder_rate
0,32240,30027,1,30,29,5.000000,72,72,5.0,31,29,4.870968,0.935484
1,32240,44144,1,52,51,4.403846,72,72,5.0,52,51,4.403846,0.980769
2,32240,22713,1,40,39,3.525000,72,72,5.0,45,40,3.511111,0.888889
3,32240,40852,1,56,55,2.482143,72,72,5.0,65,61,2.507692,0.938462
4,32240,19508,0,47,46,6.000000,72,72,5.0,90,67,8.022222,0.744444


### Features derivadas

In [29]:
model_df = (
    training_data
    .merge(
        user_product_features,
        on=["user_id", "product_id"],
        how="left"
    )
    .merge(
        user_features,
        on="user_id",
        how="left"
    )
    .merge(
        product_features,
        on="product_id",
        how="left"
    )
)

print(model_df.shape)
model_df.head()

(34508, 13)


,user_id,product_id,target,user_product_purchase_count,user_product_reorder_count,user_product_avg_add_to_cart_order,user_product_last_order_number,user_total_prior_orders,user_avg_days_since_prior_order,product_purchase_count,product_reorder_count,product_avg_add_to_cart_order,product_reorder_rate
0,32240,30027,1,30,29,5.000000,72,72,5.0,31,29,4.870968,0.935484
1,32240,44144,1,52,51,4.403846,72,72,5.0,52,51,4.403846,0.980769
2,32240,22713,1,40,39,3.525000,72,72,5.0,45,40,3.511111,0.888889
3,32240,40852,1,56,55,2.482143,72,72,5.0,65,61,2.507692,0.938462
4,32240,19508,0,47,46,6.000000,72,72,5.0,90,67,8.022222,0.744444


In [30]:
# Proporción de órdenes del usuario donde compró ese producto.
# Ejemplo:
# Si el usuario tiene 10 órdenes y compró Banana en 6,
# entonces user_product_purchase_rate = 0.6

model_df["user_product_purchase_rate"] = (
    model_df["user_product_purchase_count"] /
    model_df["user_total_prior_orders"]
)

# Tasa de reorder del producto para ese usuario.
# Si nunca fue marcado como reorder, queda 0.

model_df["user_product_reorder_rate"] = (
    model_df["user_product_reorder_count"] /
    model_df["user_product_purchase_count"]
)

# Qué tan reciente fue la última compra del producto.
# Menor valor = más reciente.
# Ejemplo:
# Si el usuario va en la orden 20 y compró el producto por última vez en la 18,
# recency = 2.

model_df["user_product_orders_since_last_purchase"] = (
    model_df["user_total_prior_orders"] -
    model_df["user_product_last_order_number"]
)

model_df.head()

,user_id,product_id,target,user_product_purchase_count,user_product_reorder_count,user_product_avg_add_to_cart_order,user_product_last_order_number,user_total_prior_orders,user_avg_days_since_prior_order,product_purchase_count,product_reorder_count,product_avg_add_to_cart_order,product_reorder_rate,user_product_purchase_rate,user_product_reorder_rate,user_product_orders_since_last_purchase
0,32240,30027,1,30,29,5.000000,72,72,5.0,31,29,4.870968,0.935484,0.416667,0.966667,0
1,32240,44144,1,52,51,4.403846,72,72,5.0,52,51,4.403846,0.980769,0.722222,0.980769,0
2,32240,22713,1,40,39,3.525000,72,72,5.0,45,40,3.511111,0.888889,0.555556,0.975000,0
3,32240,40852,1,56,55,2.482143,72,72,5.0,65,61,2.507692,0.938462,0.777778,0.982143,0
4,32240,19508,0,47,46,6.000000,72,72,5.0,90,67,8.022222,0.744444,0.652778,0.978723,0


In [31]:
#Nulos
model_df.isna().sum().sort_values(ascending=False)

user_id                                    0
product_id                                 0
target                                     0
user_product_purchase_count                0
user_product_reorder_count                 0
user_product_avg_add_to_cart_order         0
user_product_last_order_number             0
user_total_prior_orders                    0
user_avg_days_since_prior_order            0
product_purchase_count                     0
product_reorder_count                      0
product_avg_add_to_cart_order              0
product_reorder_rate                       0
user_product_purchase_rate                 0
user_product_reorder_rate                  0
user_product_orders_since_last_purchase    0
dtype: int64

In [32]:
# Para esta primera prueba, rellenamos nulos numéricos con 0.
# Más adelante podemos tratar cada variable con más cuidado.

model_df = model_df.fillna(0)

model_df.isna().sum().sum()

np.int64(0)

In [33]:
# Unimos nombres de productos solo para inspección humana.
# No usaremos product_name como feature del modelo.

model_df_named = (
    model_df
    .merge(
        products[["product_id", "product_name", "aisle_id", "department_id"]],
        on="product_id",
        how="left"
    )
)

model_df_named.head(20)

,user_id,product_id,target,user_product_purchase_count,user_product_reorder_count,user_product_avg_add_to_cart_order,user_product_last_order_number,user_total_prior_orders,user_avg_days_since_prior_order,product_purchase_count,product_reorder_count,product_avg_add_to_cart_order,product_reorder_rate,user_product_purchase_rate,user_product_reorder_rate,user_product_orders_since_last_purchase,product_name,aisle_id,department_id
0,32240,30027,1,30,29,5.000000,72,72,5.000000,31,29,4.870968,0.935484,0.416667,0.966667,0,Organic Chard Green,83,4
1,32240,44144,1,52,51,4.403846,72,72,5.000000,52,51,4.403846,0.980769,0.722222,0.980769,0,Natural Skinless & Boneless Sardines In Water,95,15
2,32240,22713,1,40,39,3.525000,72,72,5.000000,45,40,3.511111,0.888889,0.555556,0.975000,0,Ground Chicken Breast,35,12
3,32240,40852,1,56,55,2.482143,72,72,5.000000,65,61,2.507692,0.938462,0.777778,0.982143,0,Lactose Free Fat Free Milk,91,16
4,32240,19508,0,47,46,6.000000,72,72,5.000000,90,67,8.022222,0.744444,0.652778,0.978723,0,Corn Tortillas,128,3
5,32240,8996,1,35,34,5.914286,71,72,5.000000,36,34,6.444444,0.944444,0.486111,0.971429,1,Squirrelly Sprouted Grain Bread,112,3
6,32240,4476,0,17,16,10.823529,72,72,5.000000,18,16,11.333333,0.888889,0.236111,0.941176,0,Walnut Halves And Pieces,117,19
7,32240,24184,1,18,17,8.166667,72,72,5.000000,206,151,8.815534,0.733010,0.250000,0.944444,0,Red Peppers,83,4
8,32240,26209,0,14,13,9.428571,71,72,5.000000,370,265,8.027027,0.716216,0.194444,0.928571,1,Limes,24,4
9,32240,15591,0,1,0,10.000000,68,72,5.000000,3,0,13.333333,0.000000,0.013889,0.000000,4,Gluten Free Steel Cut Oats,130,14


In [34]:
#Ejemplo positivo

model_df_named[model_df_named["target"] == 1].head(20)

,user_id,product_id,target,user_product_purchase_count,user_product_reorder_count,user_product_avg_add_to_cart_order,user_product_last_order_number,user_total_prior_orders,user_avg_days_since_prior_order,product_purchase_count,product_reorder_count,product_avg_add_to_cart_order,product_reorder_rate,user_product_purchase_rate,user_product_reorder_rate,user_product_orders_since_last_purchase,product_name,aisle_id,department_id
0,32240,30027,1,30,29,5.000000,72,72,5.000000,31,29,4.870968,0.935484,0.416667,0.966667,0,Organic Chard Green,83,4
1,32240,44144,1,52,51,4.403846,72,72,5.000000,52,51,4.403846,0.980769,0.722222,0.980769,0,Natural Skinless & Boneless Sardines In Water,95,15
2,32240,22713,1,40,39,3.525000,72,72,5.000000,45,40,3.511111,0.888889,0.555556,0.975000,0,Ground Chicken Breast,35,12
3,32240,40852,1,56,55,2.482143,72,72,5.000000,65,61,2.507692,0.938462,0.777778,0.982143,0,Lactose Free Fat Free Milk,91,16
5,32240,8996,1,35,34,5.914286,71,72,5.000000,36,34,6.444444,0.944444,0.486111,0.971429,1,Squirrelly Sprouted Grain Bread,112,3
7,32240,24184,1,18,17,8.166667,72,72,5.000000,206,151,8.815534,0.733010,0.250000,0.944444,0,Red Peppers,83,4
11,34108,22935,1,12,11,3.166667,14,14,11.923077,340,243,8.991176,0.714706,0.857143,0.916667,0,Organic Yellow Onion,83,4
12,34108,20869,1,10,9,3.900000,14,14,11.923077,13,9,6.461538,0.692308,0.714286,0.900000,0,Yellow Potato,83,4
13,34108,18479,1,8,7,5.625000,14,14,11.923077,19,11,7.631579,0.578947,0.571429,0.875000,0,Organic Low Sodium Vegetable Broth,69,15
14,34108,13291,1,7,6,8.428571,14,14,11.923077,7,6,8.428571,0.857143,0.500000,0.857143,0,Organic Cabbage,83,4


In [35]:
#Ejemplo negativo

model_df_named[model_df_named["target"] == 0].head(20)

,user_id,product_id,target,user_product_purchase_count,user_product_reorder_count,user_product_avg_add_to_cart_order,user_product_last_order_number,user_total_prior_orders,user_avg_days_since_prior_order,product_purchase_count,product_reorder_count,product_avg_add_to_cart_order,product_reorder_rate,user_product_purchase_rate,user_product_reorder_rate,user_product_orders_since_last_purchase,product_name,aisle_id,department_id
4,32240,19508,0,47,46,6.000000,72,72,5.000000,90,67,8.022222,0.744444,0.652778,0.978723,0,Corn Tortillas,128,3
6,32240,4476,0,17,16,10.823529,72,72,5.000000,18,16,11.333333,0.888889,0.236111,0.941176,0,Walnut Halves And Pieces,117,19
8,32240,26209,0,14,13,9.428571,71,72,5.000000,370,265,8.027027,0.716216,0.194444,0.928571,1,Limes,24,4
9,32240,15591,0,1,0,10.000000,68,72,5.000000,3,0,13.333333,0.000000,0.013889,0.000000,4,Gluten Free Steel Cut Oats,130,14
10,34108,47485,0,7,6,3.285714,8,14,11.923077,7,6,3.285714,0.857143,0.500000,0.857143,6,Mini Chocolate Chip Waffles,52,1
15,34108,5373,0,5,4,10.000000,14,14,11.923077,27,16,10.777778,0.592593,0.357143,0.800000,0,Organic No Salt Added Diced Tomatoes,81,15
16,34108,31506,0,4,3,14.250000,14,14,11.923077,166,99,7.945783,0.596386,0.285714,0.750000,0,Extra Virgin Olive Oil,19,13
20,34108,47626,0,6,5,11.500000,14,14,11.923077,458,344,7.528384,0.751092,0.428571,0.833333,0,Large Lemon,24,4
21,18327,765,0,4,3,1.000000,13,24,10.478261,5,3,1.000000,0.600000,0.166667,0.750000,11,Swaddlers Diapers Jumbo Pack Size Newborn,56,18
22,44563,15182,0,2,1,13.000000,13,21,10.100000,6,3,10.166667,0.500000,0.095238,0.500000,8,"Organic Turbinado, Raw Cane Sugar",17,13


## Entrenamiento del Modelo

Para esta primera prueba usaremos LogisticRegression porque permite entender la relación entre features y probabilidad de recompra.

In [36]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    precision_recall_curve,
)

In [37]:
feature_cols = [
    "user_product_purchase_count",
    "user_product_reorder_count",
    "user_product_avg_add_to_cart_order",
    "user_product_last_order_number",
    "user_total_prior_orders",
    "user_avg_days_since_prior_order",
    "product_purchase_count",
    "product_reorder_count",
    "product_avg_add_to_cart_order",
    "product_reorder_rate",
    "user_product_purchase_rate",
    "user_product_reorder_rate",
    "user_product_orders_since_last_purchase",
]

X = model_df[feature_cols].copy()
y = model_df["target"].copy()

print("X:", X.shape)
print("y:", y.shape)
print(y.value_counts(normalize=True))

X: (34508, 13)
y: (34508,)
target
0    0.907761
1    0.092239
Name: proportion, dtype: float64


In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("Train target distribution:")
print(y_train.value_counts(normalize=True))

print("Test target distribution:")
print(y_test.value_counts(normalize=True))

X_train: (24155, 13)
X_test: (10353, 13)
Train target distribution:
target
0    0.907762
1    0.092238
Name: proportion, dtype: float64
Test target distribution:
target
0    0.907756
1    0.092244
Name: proportion, dtype: float64


Aquí usamos stratify=y para que train y test mantengan una proporción parecida de 0 y 1.

In [40]:
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

model.fit(X_train, y_train)

print("Model trained")

Model trained


StandardScaler:
normaliza las variables numéricas para que estén en escalas comparables.

LogisticRegression:
aprende pesos para estimar la probabilidad de que target = 1.

class_weight="balanced":
compensa parcialmente el desbalance entre productos recomprados y no recomprados.

In [41]:
## Predicciones
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Predicciones listas")

Predicciones listas


y_proba es más importante que y_pred, porque en recomendadores normalmente nos interesa ordenar productos por probabilidad.

### Metricas basicas

In [42]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.76      0.85      9398
           1       0.23      0.71      0.35       955

    accuracy                           0.75     10353
   macro avg       0.60      0.73      0.60     10353
weighted avg       0.89      0.75      0.80     10353



In [43]:
auc = roc_auc_score(y_test, y_proba)
print("ROC AUC:", auc)

ROC AUC: 0.8202108279694131


In [44]:
confusion_matrix(y_test, y_pred)

array([[7121, 2277],
       [ 281,  674]])

In [45]:
# probabilidades en filas reales

test_results = X_test.copy()
test_results["target"] = y_test.values
test_results["predicted_proba"] = y_proba
test_results["predicted_class"] = y_pred

test_results.head()

,user_product_purchase_count,user_product_reorder_count,user_product_avg_add_to_cart_order,user_product_last_order_number,user_total_prior_orders,user_avg_days_since_prior_order,product_purchase_count,product_reorder_count,product_avg_add_to_cart_order,product_reorder_rate,user_product_purchase_rate,user_product_reorder_rate,user_product_orders_since_last_purchase,target,predicted_proba,predicted_class
16472,4,3,4.500000,7,7,21.833333,205,141,8.414634,0.687805,0.571429,0.750000,0,0,0.868006,1
20811,3,2,3.333333,23,25,11.500000,121,63,8.900826,0.520661,0.120000,0.666667,2,0,0.556225,1
13187,2,1,2.000000,3,6,15.000000,340,243,8.991176,0.714706,0.333333,0.500000,3,1,0.697204,1
28693,1,0,2.000000,38,38,6.513514,79,52,8.240506,0.658228,0.026316,0.000000,0,0,0.430789,0
34431,1,0,4.000000,30,99,2.948980,3,0,5.666667,0.000000,0.010101,0.000000,69,0,0.000722,0


Como X_test perdió user_id y product_id, reconstruimos una tabla más interpretable.

In [46]:
test_results_named = (
    model_df.loc[X_test.index, ["user_id", "product_id", "target"] + feature_cols]
    .copy()
)

test_results_named["predicted_proba"] = y_proba
test_results_named["predicted_class"] = y_pred

test_results_named = (
    test_results_named
    .merge(
        products[["product_id", "product_name", "aisle_id", "department_id"]],
        on="product_id",
        how="left"
    )
)

test_results_named[
    [
        "user_id",
        "product_id",
        "product_name",
        "target",
        "predicted_proba",
        "predicted_class",
        "user_product_purchase_count",
        "user_product_purchase_rate",
        "user_product_orders_since_last_purchase",
        "product_reorder_rate",
    ]
].sort_values("predicted_proba", ascending=False).head(30)

,user_id,product_id,product_name,target,predicted_proba,predicted_class,user_product_purchase_count,user_product_purchase_rate,user_product_orders_since_last_purchase,product_reorder_rate
7616,152713,24852,Banana,1,0.983339,1,10,1.000000,0,0.876912
3658,23315,24852,Banana,0,0.982503,1,5,1.000000,0,0.876912
8850,89095,24852,Banana,1,0.982431,1,4,1.000000,0,0.876912
7041,125871,47209,Organic Hass Avocado,1,0.980432,1,10,1.000000,0,0.844981
10170,53663,24852,Banana,0,0.980268,1,24,0.827586,0,0.876912
1499,158102,1194,Natural Artisan Water,1,0.978786,1,66,0.857143,0,0.920354
9492,866,21903,Organic Baby Spinach,1,0.978106,1,22,1.000000,0,0.780612
5260,1764,39475,Total Greek Strained Yogurt,1,0.977267,1,37,0.973684,0,0.866667
9366,151154,24852,Banana,0,0.976274,1,14,0.823529,0,0.876912
7809,56878,44008,Organic Greek Whole Milk Blended Vanilla Bean ...,1,0.973762,1,36,0.900000,0,0.918033


Interpretar coeficientes del modelo

In [47]:
classifier = model.named_steps["classifier"]

coefficients = pd.DataFrame(
    {
        "feature": feature_cols,
        "coefficient": classifier.coef_[0],
    }
).sort_values("coefficient", ascending=False)

coefficients

,feature,coefficient
10,user_product_purchase_rate,0.466181
3,user_product_last_order_number,0.353759
11,user_product_reorder_rate,0.276197
9,product_reorder_rate,0.246382
2,user_product_avg_add_to_cart_order,0.044865
6,product_purchase_count,0.044712
7,product_reorder_count,0.032602
0,user_product_purchase_count,0.020903
1,user_product_reorder_count,0.020903
8,product_avg_add_to_cart_order,-0.010691


Coeficiente positivo:
cuando esa variable sube, aumenta la probabilidad estimada de recompra.

Coeficiente negativo:
cuando esa variable sube, baja la probabilidad estimada de recompra.

In [48]:
# Modelo como recomendador
def recommend_for_user(user_id, top_k=10):
    """
    Genera recomendaciones para un usuario usando el modelo supervisado.

    Toma todos los productos candidatos del usuario en model_df,
    calcula probabilidad de recompra y retorna los top_k productos.
    """
    
    user_candidates = model_df[model_df["user_id"] == user_id].copy()
    
    if user_candidates.empty:
        return pd.DataFrame()
    
    X_user = user_candidates[feature_cols].copy()
    user_candidates["predicted_proba"] = model.predict_proba(X_user)[:, 1]
    
    user_recommendations = (
        user_candidates
        .sort_values("predicted_proba", ascending=False)
        .head(top_k)
        .merge(
            products[["product_id", "product_name", "aisle_id", "department_id"]],
            on="product_id",
            how="left"
        )
    )
    
    return user_recommendations[
        [
            "user_id",
            "product_id",
            "product_name",
            "target",
            "predicted_proba",
            "user_product_purchase_count",
            "user_product_purchase_rate",
            "user_product_orders_since_last_purchase",
            "product_reorder_rate",
        ]
    ]

In [49]:
sample_user = model_df["user_id"].iloc[0]

recommend_for_user(sample_user, top_k=10)

,user_id,product_id,product_name,target,predicted_proba,user_product_purchase_count,user_product_purchase_rate,user_product_orders_since_last_purchase,product_reorder_rate
0,32240,40852,Lactose Free Fat Free Milk,1,0.970356,56,0.777778,0,0.938462
1,32240,44144,Natural Skinless & Boneless Sardines In Water,1,0.965257,52,0.722222,0,0.980769
2,32240,24852,Banana,1,0.959712,37,0.513889,0,0.876912
3,32240,24253,Go Lean Cereal,1,0.954573,48,0.666667,0,0.885714
4,32240,14462,Sliced Black Olives,1,0.951437,47,0.652778,0,0.859375
5,32240,19508,Corn Tortillas,0,0.947249,47,0.652778,0,0.744444
6,32240,22713,Ground Chicken Breast,1,0.932974,40,0.555556,0,0.888889
7,32240,8996,Squirrelly Sprouted Grain Bread,1,0.912081,35,0.486111,1,0.944444
8,32240,17795,Organic Green Leaf Lettuce,0,0.897784,50,0.694444,10,0.817204
9,32240,30027,Organic Chard Green,1,0.897073,30,0.416667,0,0.935484


target = 1 significa que el usuario sí compró ese producto en la orden train.
predicted_proba = probabilidad estimada por el modelo.

- Para aprendizaje, lo ideal es mirar si los productos con target = 1 aparecen arriba.

In [50]:
sample_users_eval = model_df["user_id"].drop_duplicates().head(20).tolist()

for user_id in sample_users_eval[:5]:
    print("=" * 80)
    print("User:", user_id)
    display(recommend_for_user(user_id, top_k=10))

User: 32240


,user_id,product_id,product_name,target,predicted_proba,user_product_purchase_count,user_product_purchase_rate,user_product_orders_since_last_purchase,product_reorder_rate
0,32240,40852,Lactose Free Fat Free Milk,1,0.970356,56,0.777778,0,0.938462
1,32240,44144,Natural Skinless & Boneless Sardines In Water,1,0.965257,52,0.722222,0,0.980769
2,32240,24852,Banana,1,0.959712,37,0.513889,0,0.876912
3,32240,24253,Go Lean Cereal,1,0.954573,48,0.666667,0,0.885714
4,32240,14462,Sliced Black Olives,1,0.951437,47,0.652778,0,0.859375
5,32240,19508,Corn Tortillas,0,0.947249,47,0.652778,0,0.744444
6,32240,22713,Ground Chicken Breast,1,0.932974,40,0.555556,0,0.888889
7,32240,8996,Squirrelly Sprouted Grain Bread,1,0.912081,35,0.486111,1,0.944444
8,32240,17795,Organic Green Leaf Lettuce,0,0.897784,50,0.694444,10,0.817204
9,32240,30027,Organic Chard Green,1,0.897073,30,0.416667,0,0.935484


User: 34108


,user_id,product_id,product_name,target,predicted_proba,user_product_purchase_count,user_product_purchase_rate,user_product_orders_since_last_purchase,product_reorder_rate
0,34108,22935,Organic Yellow Onion,1,0.955965,12,0.857143,0,0.714706
1,34108,20869,Yellow Potato,1,0.922502,10,0.714286,0,0.692308
2,34108,45007,Organic Zucchini,1,0.918748,9,0.642857,0,0.736486
3,34108,24852,Banana,1,0.902660,5,0.357143,0,0.876912
4,34108,45063,Organic Baby Bella Mushrooms,1,0.900980,9,0.642857,0,0.625000
5,34108,21137,Organic Strawberries,1,0.891086,6,0.428571,0,0.822671
6,34108,13291,Organic Cabbage,1,0.876532,7,0.500000,0,0.857143
7,34108,18479,Organic Low Sodium Vegetable Broth,1,0.874538,8,0.571429,0,0.578947
8,34108,24184,Red Peppers,1,0.873714,7,0.500000,0,0.733010
9,34108,47626,Large Lemon,0,0.866167,6,0.428571,0,0.751092


User: 18327


,user_id,product_id,product_name,target,predicted_proba,user_product_purchase_count,user_product_purchase_rate,user_product_orders_since_last_purchase,product_reorder_rate
0,18327,8475,Diet 12 Oz Ginger Ale,1,0.783880,7,0.291667,0,0.777778
1,18327,277,Diet Root Beer,1,0.762402,6,0.250000,0,0.800000
2,18327,24852,Banana,1,0.756683,5,0.208333,7,0.876912
3,18327,25443,"Delights Turkey Sausage, Egg Whites & Cheese C...",0,0.708152,5,0.208333,0,0.666667
4,18327,4778,Wheat Thins Reduced Fat Crackers,0,0.664330,4,0.166667,2,0.807692
5,18327,45478,Vanilla With Caramel Low Fat Ice Cream Cone,0,0.638662,3,0.125000,2,0.913043
6,18327,37646,Organic Gala Apples,0,0.591805,3,0.125000,3,0.736842
7,18327,2609,Newborn Bottles with Iron Infant Formula,0,0.584089,4,0.166667,4,0.666667
8,18327,47209,Organic Hass Avocado,0,0.580230,3,0.125000,8,0.844981
9,18327,34358,Garlic,0,0.531240,2,0.083333,1,0.563636


User: 44563


,user_id,product_id,product_name,target,predicted_proba,user_product_purchase_count,user_product_purchase_rate,user_product_orders_since_last_purchase,product_reorder_rate
0,44563,22935,Organic Yellow Onion,1,0.940694,15,0.714286,0,0.714706
1,44563,24964,Organic Garlic,1,0.937134,15,0.714286,0,0.675182
2,44563,41596,Frozen Blueberries,1,0.935014,14,0.666667,0,0.941176
3,44563,34525,Organic Vanilla Whole Milk Yogurt,1,0.931018,14,0.666667,0,0.871795
4,44563,27156,Organic Black Beans,1,0.913438,15,0.714286,1,0.518987
5,44563,33000,Pure Irish Butter,0,0.906670,12,0.571429,0,0.760000
6,44563,48079,Organic Brown Long Grain Rice,1,0.895169,12,0.571429,1,0.807692
7,44563,47823,Organic Vanilla Kefir,1,0.885432,10,0.476190,0,0.900000
8,44563,26695,Garlic Marinara Pasta Sauce,0,0.872957,10,0.476190,0,0.785714
9,44563,5876,Organic Lemon,0,0.852144,8,0.380952,0,0.778135


User: 136134


,user_id,product_id,product_name,target,predicted_proba,user_product_purchase_count,user_product_purchase_rate,user_product_orders_since_last_purchase,product_reorder_rate
0,136134,27845,Organic Whole Milk,1,0.978810,56,0.848485,0,0.869748
1,136134,24852,Banana,0,0.964846,37,0.560606,0,0.876912
2,136134,25146,Original Orange Juice,0,0.958506,47,0.712121,0,0.817460
3,136134,43394,Organic Lactose Free Whole Milk,1,0.957851,47,0.712121,1,0.942029
4,136134,48186,Lactose Free Half & Half,0,0.950233,42,0.636364,0,0.955556
5,136134,21137,Organic Strawberries,1,0.943740,34,0.515152,0,0.822671
6,136134,30635,Lactose Free Yogurt Plain,1,0.929962,37,0.560606,1,0.948718
7,136134,17200,Wild Blueberry Preserves,0,0.879240,34,0.515152,4,0.755556
8,136134,33642,Lactose Free Cream Cheese,1,0.859645,22,0.333333,0,0.888889
9,136134,2763,Uncured Hickory Smoked Bacon,0,0.858570,23,0.348485,0,0.800000


In [51]:
# Evaluar ranking simple por usuario

sample_users_eval = model_df["user_id"].drop_duplicates().head(20).tolist()

for user_id in sample_users_eval[:5]:
    print("=" * 80)
    print("User:", user_id)
    display(recommend_for_user(user_id, top_k=10))

User: 32240


,user_id,product_id,product_name,target,predicted_proba,user_product_purchase_count,user_product_purchase_rate,user_product_orders_since_last_purchase,product_reorder_rate
0,32240,40852,Lactose Free Fat Free Milk,1,0.970356,56,0.777778,0,0.938462
1,32240,44144,Natural Skinless & Boneless Sardines In Water,1,0.965257,52,0.722222,0,0.980769
2,32240,24852,Banana,1,0.959712,37,0.513889,0,0.876912
3,32240,24253,Go Lean Cereal,1,0.954573,48,0.666667,0,0.885714
4,32240,14462,Sliced Black Olives,1,0.951437,47,0.652778,0,0.859375
5,32240,19508,Corn Tortillas,0,0.947249,47,0.652778,0,0.744444
6,32240,22713,Ground Chicken Breast,1,0.932974,40,0.555556,0,0.888889
7,32240,8996,Squirrelly Sprouted Grain Bread,1,0.912081,35,0.486111,1,0.944444
8,32240,17795,Organic Green Leaf Lettuce,0,0.897784,50,0.694444,10,0.817204
9,32240,30027,Organic Chard Green,1,0.897073,30,0.416667,0,0.935484


User: 34108


,user_id,product_id,product_name,target,predicted_proba,user_product_purchase_count,user_product_purchase_rate,user_product_orders_since_last_purchase,product_reorder_rate
0,34108,22935,Organic Yellow Onion,1,0.955965,12,0.857143,0,0.714706
1,34108,20869,Yellow Potato,1,0.922502,10,0.714286,0,0.692308
2,34108,45007,Organic Zucchini,1,0.918748,9,0.642857,0,0.736486
3,34108,24852,Banana,1,0.902660,5,0.357143,0,0.876912
4,34108,45063,Organic Baby Bella Mushrooms,1,0.900980,9,0.642857,0,0.625000
5,34108,21137,Organic Strawberries,1,0.891086,6,0.428571,0,0.822671
6,34108,13291,Organic Cabbage,1,0.876532,7,0.500000,0,0.857143
7,34108,18479,Organic Low Sodium Vegetable Broth,1,0.874538,8,0.571429,0,0.578947
8,34108,24184,Red Peppers,1,0.873714,7,0.500000,0,0.733010
9,34108,47626,Large Lemon,0,0.866167,6,0.428571,0,0.751092


User: 18327


,user_id,product_id,product_name,target,predicted_proba,user_product_purchase_count,user_product_purchase_rate,user_product_orders_since_last_purchase,product_reorder_rate
0,18327,8475,Diet 12 Oz Ginger Ale,1,0.783880,7,0.291667,0,0.777778
1,18327,277,Diet Root Beer,1,0.762402,6,0.250000,0,0.800000
2,18327,24852,Banana,1,0.756683,5,0.208333,7,0.876912
3,18327,25443,"Delights Turkey Sausage, Egg Whites & Cheese C...",0,0.708152,5,0.208333,0,0.666667
4,18327,4778,Wheat Thins Reduced Fat Crackers,0,0.664330,4,0.166667,2,0.807692
5,18327,45478,Vanilla With Caramel Low Fat Ice Cream Cone,0,0.638662,3,0.125000,2,0.913043
6,18327,37646,Organic Gala Apples,0,0.591805,3,0.125000,3,0.736842
7,18327,2609,Newborn Bottles with Iron Infant Formula,0,0.584089,4,0.166667,4,0.666667
8,18327,47209,Organic Hass Avocado,0,0.580230,3,0.125000,8,0.844981
9,18327,34358,Garlic,0,0.531240,2,0.083333,1,0.563636


User: 44563


,user_id,product_id,product_name,target,predicted_proba,user_product_purchase_count,user_product_purchase_rate,user_product_orders_since_last_purchase,product_reorder_rate
0,44563,22935,Organic Yellow Onion,1,0.940694,15,0.714286,0,0.714706
1,44563,24964,Organic Garlic,1,0.937134,15,0.714286,0,0.675182
2,44563,41596,Frozen Blueberries,1,0.935014,14,0.666667,0,0.941176
3,44563,34525,Organic Vanilla Whole Milk Yogurt,1,0.931018,14,0.666667,0,0.871795
4,44563,27156,Organic Black Beans,1,0.913438,15,0.714286,1,0.518987
5,44563,33000,Pure Irish Butter,0,0.906670,12,0.571429,0,0.760000
6,44563,48079,Organic Brown Long Grain Rice,1,0.895169,12,0.571429,1,0.807692
7,44563,47823,Organic Vanilla Kefir,1,0.885432,10,0.476190,0,0.900000
8,44563,26695,Garlic Marinara Pasta Sauce,0,0.872957,10,0.476190,0,0.785714
9,44563,5876,Organic Lemon,0,0.852144,8,0.380952,0,0.778135


User: 136134


,user_id,product_id,product_name,target,predicted_proba,user_product_purchase_count,user_product_purchase_rate,user_product_orders_since_last_purchase,product_reorder_rate
0,136134,27845,Organic Whole Milk,1,0.978810,56,0.848485,0,0.869748
1,136134,24852,Banana,0,0.964846,37,0.560606,0,0.876912
2,136134,25146,Original Orange Juice,0,0.958506,47,0.712121,0,0.817460
3,136134,43394,Organic Lactose Free Whole Milk,1,0.957851,47,0.712121,1,0.942029
4,136134,48186,Lactose Free Half & Half,0,0.950233,42,0.636364,0,0.955556
5,136134,21137,Organic Strawberries,1,0.943740,34,0.515152,0,0.822671
6,136134,30635,Lactose Free Yogurt Plain,1,0.929962,37,0.560606,1,0.948718
7,136134,17200,Wild Blueberry Preserves,0,0.879240,34,0.515152,4,0.755556
8,136134,33642,Lactose Free Cream Cheese,1,0.859645,22,0.333333,0,0.888889
9,136134,2763,Uncured Hickory Smoked Bacon,0,0.858570,23,0.348485,0,0.800000


Input:
user_id + product_id + features históricas

Output:
probabilidad de recompra

Uso:
ordenar productos candidatos y recomendar los más probables